# Bankruptcy Risk Prediction — Independent Reconstruction

An original, reproducible reconstruction inspired by WorldQuant University Applied Data Science Lab Project 5. **This is not a recovered course submission.** It uses the publicly licensed UCI Polish Companies Bankruptcy dataset and compares Random Forest with Gradient Boosting on an imbalanced classification task.

Run all cells to generate your own results. No metrics are claimed until executed.

## 1. Load data and inspect imbalance

UCI dataset 365, CC BY 4.0, Tomczak (2016). We use only the **3-year cohort** so that different forecast horizons are not mixed. `year` is excluded to avoid a redundant constant feature.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score, classification_report, ConfusionMatrixDisplay, PrecisionRecallDisplay

RANDOM_STATE = 42
dataset = fetch_ucirepo(id=365)
X_all = dataset.data.features.copy()
y_all = dataset.data.targets.squeeze().astype(int)
mask = X_all["year"].eq(3)
X = X_all.loc[mask].drop(columns="year").replace([np.inf, -np.inf], np.nan)
y = y_all.loc[mask]
print(f"Observations: {len(X):,}; predictors: {X.shape[1]}; bankrupt: {int(y.sum()):,} ({y.mean():.2%})")
print("Missing values:", int(X.isna().sum().sum()))
y.value_counts().sort_index().rename(index={0:"Not bankrupt",1:"Bankrupt"}).plot.bar(title="Class distribution")
plt.ylabel("Companies")
plt.tight_layout()
plt.show()

## 2. Holdout split

Use a stratified split before any resampling. Never oversample the test set or impute using test-set statistics.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print("Training:", X_train.shape, "Test:", X_test.shape)

## 3. Random Forest and Gradient Boosting

The models share the same median-imputation strategy. Training-only random oversampling is done **inside** the pipeline, preventing resampling leakage into the held-out test set. The default 0.5 decision threshold is used initially; business costs may warrant a different threshold.

In [ ]:
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler

models = {
    "Random Forest": ImbPipeline([
        ("impute", SimpleImputer(strategy="median", add_indicator=True)),
        ("oversample", RandomOverSampler(random_state=RANDOM_STATE)),
        ("classifier", RandomForestClassifier(n_estimators=200, min_samples_leaf=3, random_state=RANDOM_STATE, n_jobs=-1)),
    ]),
    "Gradient Boosting": ImbPipeline([
        ("impute", SimpleImputer(strategy="median", add_indicator=True)),
        ("oversample", RandomOverSampler(random_state=RANDOM_STATE)),
        ("classifier", HistGradientBoostingClassifier(max_iter=150, max_leaf_nodes=15, learning_rate=0.05, random_state=RANDOM_STATE)),
    ]),
}
results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    predicted = model.predict(X_test)
    probability = model.predict_proba(X_test)[:, 1]
    results.append({
        "Model": name,
        "Precision": precision_score(y_test, predicted, zero_division=0),
        "Recall": recall_score(y_test, predicted, zero_division=0),
        "F1": f1_score(y_test, predicted, zero_division=0),
        "Average precision (PR-AUC)": average_precision_score(y_test, probability),
    })
    print(f"\n{name}\n", classification_report(y_test, predicted, zero_division=0))
    ConfusionMatrixDisplay.from_predictions(y_test, predicted, display_labels=["Healthy", "Bankrupt"], cmap="Blues")
    plt.title(name + " — confusion matrix")
    plt.show()

pd.DataFrame(results).set_index("Model").round(3)

## 4. Precision–recall curves and error trade-offs

With rare bankruptcies, accuracy can be misleading. Recall captures the fraction of bankrupt companies found; precision captures how many flagged companies actually went bankrupt. False positives can impose costly unnecessary investigations.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for name, model in models.items():
    PrecisionRecallDisplay.from_estimator(model, X_test, y_test, name=name, ax=ax)
ax.set_title("Precision–recall comparison (held-out test set)")
plt.tight_layout()
plt.show()

## 5. Model interpretation

Feature importance is descriptive, not causal. The Random Forest below ranks its most influential imputed financial ratios. Indicators for missingness may also appear.

In [ ]:
rf = models["Random Forest"]
feature_names = rf.named_steps["impute"].get_feature_names_out(X.columns)
importances = pd.Series(rf.named_steps["classifier"].feature_importances_, index=feature_names)
importances.nlargest(12).sort_values().plot.barh(figsize=(8, 5), title="Random Forest feature importance")
plt.tight_layout()
plt.show()

## 6. Limitations and next steps

- This is a single historical cohort; performance may change for other years, countries, or time periods.
- A single held-out split is an initial evaluation, not a guarantee of real-world performance.
- Resampling changes the training distribution; predicted probabilities may require calibration.
- For production use: tune thresholds against investigation cost, use cross-validation on training data, monitor data drift and false positives, and test fairness where appropriate.

**Data citation:** Tomczak, S. (2016). *Polish Companies Bankruptcy*. UCI Machine Learning Repository. DOI: 10.24432/C5F600.

**Learning inspiration:** WorldQuant University, Applied Data Science Lab, Project 5. This notebook is an independent reconstruction, not a copy of course solutions.